In [ ]:
#Installs
!pip install umap-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 2.6 MB/s eta 0:00:00


In [ ]:
#IMPORTS
import plotly.express as px
import pandas as pd
import numpy as np
import os
import glob
from gensim.models import Word2Vec
import hdbscan
import umap.umap_ as umap
import matplotlib.pyplot as plt

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
from google.colab import drive
drive.mount("/content/drive") #Datasets were uploaded to Google Drive and fetched from there

Mounted at /content/drive


In [ ]:
def create_dataframe(directory):
    causes = []
    effects = []

    #Glob to get all CSV files in the directory
    csv_files = glob.glob(os.path.join(directory, '*.csv'))

    for file in csv_files:
        try:
            df = pd.read_csv(file)

            #Check if "cause" and "effect" columns exist (lowercase)
            if 'cause' in df.columns and 'effect' in df.columns:
                # Replace missing values with 'NA' and convert to strings
                df['cause'] = df['cause'].fillna('NA').astype(str)
                df['effect'] = df['effect'].fillna('NA').astype(str)

                #Append the "cause" and "effect" data to the lists
                causes.extend(df['cause'].tolist())
                effects.extend(df['effect'].tolist())
            else:
                print(f"Skipping file {file} as it doesn't contain 'cause' and 'effect' columns")
        except Exception as e:
            print(f"Error reading file {file}: {e}")

    #Create a Dataframe from the causes and effects with columns "Cause" and "Effect"
    data = pd.DataFrame({'Cause': causes, 'Effect': effects})
    return data

data = create_dataframe('/content/drive/MyDrive/extracted_causes_effects/ThirdDataset_PDF_cause-effect') #Change dataset if needed

#Inspect dataset if needed:
# data.head(50)

# data["Cause"].str.len()

# data["Effect"].str.len().mean()

,Cause,Effect
0,##words : Ostracism Social exclusion Psychosoc...,Gender differences
1,disturbances.,","
2,has been shown both to relate to psychosocial ...,hypothalamus pituitary ad reno cortical axis (...
3,##c,does not affect psychological responses to pub...
4,- women,a blunted cortisol stress response
5,- experience of social exclusion,blunted cortisol response to stress in women b...
6,factor might,higher vulnerability to social triggers of hea...
7,ameliorates the psychological impact of stress...,support has several protective effects
8,support,##k of
9,disturbances,","


In [ ]:
causes = data['Cause'].to_list()
effects = data['Effect'].to_list()

In [ ]:
%%time
from sentence_transformers import SentenceTransformer, util

#Initialize pre-trained embedding model
model_embedding = SentenceTransformer('all-MiniLM-L6-v2')

#Encode causes and effects
cause_embeddings = model_embedding.encode(causes)
effect_embeddings = model_embedding.encode(effects)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

CPU times: user 1min 47s, sys: 1.24 s, total: 1min 48s
Wall time: 1min 56s


In [ ]:
%%time
from umap import UMAP

#Initialize UMAP model
umap_model = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine')

#Reduce dimensions for causes and effects
reduced_cause_embeddings = umap_model.fit_transform(cause_embeddings)
reduced_effect_embeddings = umap_model.fit_transform(effect_embeddings)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


CPU times: user 1min 13s, sys: 684 ms, total: 1min 13s
Wall time: 58.5 s


In [ ]:
#HDBSCAN parameters
hdbscan_params = {
    'min_cluster_size': 10,
    'metric': 'euclidean'
}

#Cluster causes and effects
cause_clusterer = hdbscan.HDBSCAN(**hdbscan_params)
cause_clusterer.fit(reduced_cause_embeddings)

effect_clusterer = hdbscan.HDBSCAN(**hdbscan_params)
effect_clusterer.fit(reduced_effect_embeddings)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


HDBSCAN(min_cluster_size=10)

In [ ]:
#Print the number of clusters

print(np.unique(cause_clusterer.labels_))

[ -1   0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16
  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34
  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52
  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70
  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88
  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106
 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124
 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142
 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157]


In [ ]:
#Prepare dataFrames for plotly for visualization
data_vis_causes = pd.DataFrame({
    'UMAP1': reduced_cause_embeddings[:, 0],
    'UMAP2': reduced_cause_embeddings[:, 1],
    'Cluster': cause_clusterer.labels_,
    'Text': causes
})

data_vis_effects = pd.DataFrame({
    'UMAP1': reduced_effect_embeddings[:, 0],
    'UMAP2': reduced_effect_embeddings[:, 1],
    'Cluster': effect_clusterer.labels_,
    'Text': effects
})

#Remove noise points from visualization (Cluster -1)
data_vis_causes_filtered = data_vis_causes[data_vis_causes['Cluster'] != -1]
data_vis_effects_filtered = data_vis_effects[data_vis_effects['Cluster'] != -1]

In [ ]:
#Plotting causes
#Font sizes have been increased to show the labels better in the thesis

fig_causes = px.scatter(
    data_vis_causes_filtered,
    x='UMAP1',
    y='UMAP2',
    color='Cluster',
    hover_data={'Text': True, 'Cluster': True},
    title="HDBSCAN Clusters for Causes in Reduced UMAP Space",
    labels={'UMAP1': 'UMAP Dimension 1', 'UMAP2': 'UMAP Dimension 2'},
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig_causes.update_traces(marker=dict(size=8, opacity=0.7))
fig_causes.update_layout(
    title_font_size=16,
    title_font=dict(size=18, family='Arial'),  #Title font
    legend_title="Cluster Label",
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    margin=dict(l=40, r=40, t=40, b=40),
    xaxis_title_font=dict(size=25, family = "Times New Roman"),  #x-axis title font size
    yaxis_title_font=dict(size=25, family = "Times New Roman"),  #y-axis title font size
    xaxis_tickfont=dict(size=20, family = "Times New Roman"),    #x-axis tick label size
    yaxis_tickfont=dict(size=20, family = "Times New Roman"),    #y-axis tick label size
    coloraxis_colorbar=dict(
        tickfont=dict(size=20, family = "Times New Roman"),      #Font size of color bar numbers
        title_font=dict(size=25, family = "Times New Roman")    #Color bar title font
    )
)
fig_causes.show()

#Plotting effects
fig_effects = px.scatter(
    data_vis_effects_filtered,
    x='UMAP1',
    y='UMAP2',
    color='Cluster',
    hover_data={'Text': True, 'Cluster': True},
    title="HDBSCAN Clusters for Effects in Reduced UMAP Space",
    labels={'UMAP1': 'UMAP Dimension 1', 'UMAP2': 'UMAP Dimension 2'},
    color_discrete_sequence=px.colors.qualitative.Pastel1
)
fig_effects.update_traces(marker=dict(size=8, opacity=0.7))
fig_effects.update_layout(
    title_font_size=16,
    title_font=dict(size=18, family='Arial'),  #Title font
    legend_title="Cluster Label",
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
    margin=dict(l=40, r=40, t=40, b=40),
    xaxis_title_font=dict(size=25, family = "Times New Roman"),  #x-axis title font size
    yaxis_title_font=dict(size=25, family = "Times New Roman"),  #y-axis title font size
    xaxis_tickfont=dict(size=20, family = "Times New Roman"),    #x-axis tick label size
    yaxis_tickfont=dict(size=20, family = "Times New Roman"),    #y-axis tick label size
    coloraxis_colorbar=dict(
        tickfont=dict(size=20, family = "Times New Roman"),      #Font size of color bar numbers
        title_font=dict(size=25, family = "Times New Roman")    #Color bar title font
    )
)
fig_effects.show()


In [ ]:
from sklearn.metrics import davies_bouldin_score, silhouette_score, calinski_harabasz_score

#METRICS

#Evaluation for causes
print("Evaluation Metrics for Causes Clustering:")
cause_labels = cause_clusterer.labels_  #Extract cluster labels
cause_embeddings_filtered = reduced_cause_embeddings[cause_labels != -1]  #Remove noise points
cause_labels_filtered = cause_labels[cause_labels != -1]  #Remove noise labels

if len(set(cause_labels_filtered)) > 1:  #Make sure there is more than one cluster existing
    db_score = davies_bouldin_score(cause_embeddings_filtered, cause_labels_filtered)
    silhouette = silhouette_score(cause_embeddings_filtered, cause_labels_filtered)
    ch_score = calinski_harabasz_score(cause_embeddings_filtered, cause_labels_filtered)

    print(f"Davies-Bouldin Index: {db_score:.3f} (Lower is better)")
    print(f"Silhouette Score: {silhouette:.3f} (Higher is better)")
    print(f"Calinski-Harabasz Score: {ch_score:.3f} (Higher is better)")
else:
    print("Not enough clusters to compute metrics")

#Evaluation for effects
print("\nEvaluation Metrics for Effects Clustering:")
effect_labels = effect_clusterer.labels_
effect_embeddings_filtered = reduced_effect_embeddings[effect_labels != -1]
effect_labels_filtered = effect_labels[effect_labels != -1]

if len(set(effect_labels_filtered)) > 1:
    db_score = davies_bouldin_score(effect_embeddings_filtered, effect_labels_filtered)
    silhouette = silhouette_score(effect_embeddings_filtered, effect_labels_filtered)
    ch_score = calinski_harabasz_score(effect_embeddings_filtered, effect_labels_filtered)

    print(f"Davies-Bouldin Index: {db_score:.3f} (Lower is better)")
    print(f"Silhouette Score: {silhouette:.3f} (Higher is better)")
    print(f"Calinski-Harabasz Score: {ch_score:.3f} (Higher is better)")
else:
    print("Not enough clusters to compute metrics")

Evaluation Metrics for Causes Clustering:
Davies-Bouldin Index: 0.378 (Lower is better)
Silhouette Score: 0.680 (Higher is better)
Calinski-Harabasz Score: 118454.016 (Higher is better)
